<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Exact matches and rules

**[Exact matches and rules](https://learning.nextia-ai.com/courses/search/m03/exact-matches-and-rules/)** · Search and Ranking: Find the Best Match · Module 3, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find product codes with an exact-match rule, measure it, and see when a rule fires on the wrong query.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 10 MB). No account, no key and no model download. |
| **You should know** | BM25 and fields, from [BM25](https://learning.nextia-ai.com/courses/search/m03/bm25/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m03/exact-matches-and-rules/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M03-L03-exact-matches-and-rules/exact-matches-and-rules-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 10 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Why BM25 cannot find a product code

Some people type a product code, such as "LK-91857": they copied it from an email or a receipt. Run the next cell. It shows the code queries of the course, and what BM25 sees.

> **Predict.** Can BM25 find product LK-91857 for the query "LK-91857"?

In [ ]:
import re
def tokens(text):
    return [t for t in re.split(r"[^\w]+|_", str(text).lower()) if t]

cat = catalogue()
q = queries()
codes = q[q["source"] == "code"]
print(len(codes), "code queries in train, validation and test; forms:", codes["query"].head(4).tolist())
text = (cat["title"] + " " + cat["description"]).str.lower()
print("tokens of 'LK-91857':", tokens("LK-91857"))
print("products whose text has '91857':", int(text.str.contains("91857").sum()))

> **Check.** No product's title or description contains "91857". The code is in the `product_id` column, which is not in the index. BM25 can only find words that are in the text.

## 2. A rule: look up the code directly

An **exact-match rule** recognises a pattern in the query and goes straight to the answer. Here, the pattern is a code: "LK" (optional, with or without a space or a dash) and five digits. If the product exists, it goes first. Run the next cell: the rule on six texts.

In [ ]:
CODE = re.compile(r"\b(?:lk[\s-]?)?(\d{5})\b", re.IGNORECASE)
known = set(cat["product_id"])

def code_matches(query):
    """The products whose code is in the query (they go first)."""
    found = []
    for m in CODE.finditer(query):
        pid = f"LK-{m.group(1)}"
        if pid in known and pid not in found:
            found.append(pid)
    return found

for text in ["LK-91857", "lk91857", "lk 91857", "91857", "bunk bed 91857", "LK-00001"]:
    print(f"{text!r:18} -> {code_matches(text)}")

> **Check.** The four ways people write the code all find LK-91857. "LK-00001" finds nothing, because no product has that ID.

## 3. Measure the rule on the code queries

In [ ]:
j = judgments()
run = {qid: code_matches(text) for qid, text in zip(codes["query_id"], codes["query"])}
pq = cp.per_query(run, j)
found = sum(1 for r in run.values() if r)
print(f"the rule finds a product for {found} of {len(codes)} code queries; NDCG@10 {pq['ndcg@10'].mean():.3f}")

> **Check.** The rule finds the product for 40 of 40 code queries, and NDCG@10 is 1.000: the one right product is first every time.

## 4. The trap: numbers that are not codes

The rule accepts a five-digit number without "LK". But many queries have numbers: sizes, power, light. Run the next cell.

In [ ]:
share = len(known) / 90000      # five-digit numbers: 10000 to 99999
print(f"{len(known):,} products: {share:.1%} of all five-digit numbers are real product codes")
traps = ["12000 btu air conditioner", "10000 lumen led flood light", "24000 btu mini split"]
for text in traps:
    m = code_matches(text)
    title = cat.set_index("product_id").loc[m[0], "title"] if m else ""
    print(f"{text!r:32} -> {m} {title}")

> **Check.** 44.5% of all five-digit numbers are real product codes. So a query such as "12000 btu air conditioner" puts an unrelated product first. These three queries are examples that we wrote; they are not in the course's query set.

## 5. Your turn: a stricter rule

Write `strict_matches`: it accepts a code with "lk" before it anywhere in the query, but a bare five-digit number **only when it is the whole query**. It must still find all 40 code queries, and find nothing for the three traps.

In [ ]:
LK = re.compile(r"\blk[\s-]?(\d{5})\b", re.IGNORECASE)
ONLY = re.compile(r"^\s*(\d{5})\s*$")

def strict_matches(query):
    """A code with its LK prefix anywhere, or a five-digit number when it is the whole query."""
    found = []
    for number in LK.findall(query) + ONLY.findall(query):
        pid = f"LK-{number}"
        if pid in known and pid not in found:
            found.append(pid)
    return found

strict_found = sum(1 for text in codes["query"] if strict_matches(text))
strict_traps = [strict_matches(text) for text in traps]
print(strict_found, strict_traps)

In [ ]:
expect('strict_found', strict_found, 40)

In [ ]:
expect_hash('strict_traps', strict_traps, 'f865256c77f3fc9f')

## 6. Numbers that are not codes

Sizes and model numbers are not product codes, but they are exact too. Run the next cell: how the tokenizer splits them, and how many queries have a number.

In [ ]:
examples = ["zachary 72.5", "3 1/2 inch drawer pull", "blk 18x18 seat cushions", 'fawkes 36" blue vanity']
for text in examples:
    print(f"{text!r:28} tokens {tokens(text)}")
nums = q[(q["has_number"]) & (q["source"] != "code")]
print(len(nums), "queries have a number and are not codes; by source:", nums["source"].value_counts().to_dict())

> **Check.** "72.5" becomes two tokens, "72" and "5", and "18x18" stays one token. 42 queries have a number and are not codes. A rule for sizes needs more care than a rule for codes: the lesson shows what BM25 does with them.

In the project: `search query "lk 91857" --run bm25-fields+codes` and `search compare bm25 bm25-fields+codes --by source`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Normalise text](https://learning.nextia-ai.com/courses/search/m04/normalise-text/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m03/exact-matches-and-rules/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.